# Environment test -- present-day SHMF and radial distribution

Paper 3 (assembly bias): comparing the three formation-time cases (**early**, **middle**, **late** -- see `z50_vr_scale_test` project notes) across the eight vr-scaling models (**fid** plus A = **0.00, 0.12, 0.25, 0.37, 0.50, 0.62, 0.75**), each evolved from `data/local_trees/environment_test/<case>/tree_<model>_evo.npz`.

Four present-day/history diagnostics:
- subhalo mass function (SHMF), $N$ per log-mass bin, down to $10^{9}\,M_\odot$ -- one panel per case, same color per model in every panel
- radial distribution of surviving subhalos, $N(>r/R_{\rm vir})$, cumulative counts on a log y-axis -- one panel per case, same color per model in every panel
- $N_{\rm sub}$ and $f_{\rm sub}$ (subhalo mass fraction) vs A -- one line per case, `fid` shown as a reference point at A=0
- each case's host MAH vs the 1e13 sample mean, legend labeled by `rat = (1+z50)/(1+\langle z50\rangle)`

All use `Tree_Reader_Light` (SatGen/mcmc/src/jsm_stellarhalo.py) for the z=0 subhalo census, and `jsm_stats.count_straight` / `jsm_stats.radii_grthan` for the binning -- same tools used elsewhere in paper 3 (see `SHMF.ipynb`), not reimplemented here.

**Caveat:** each case/model here is a *single* tree (not a 1000-tree ensemble like the other paper-3 SHMF plots), and only subhalos above $10^{9}\,M_\odot$ survive to z=0 within $R_{\rm vir}$ -- so these are noisier, single-realization comparisons, not ensemble statistics.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, '/Users/jsmonzon/Research/SatGen/mcmc/src/')
sys.path.insert(0, '/Users/jsmonzon/Research/SatGen/src/')
import jsm_stellarhalo as jsh
import jsm_stats

import warnings; warnings.simplefilter('ignore')

In [ ]:
plt.style.use('../../../SatGen/notebooks/paper1/paper.mplstyle')
double_textwidth = 7.0  # inches
single_textwidth = 3.5  # inches

## Load the 24 trees

One `Tree_Reader_Light` per (case, model) pair -- 3 cases x 8 models (fid + 7 A-scaled). `mass_threshold=1e9` sets the z=0 mass floor used for the SHMF, radial-distribution, and Nsub/fsub samples below (SatGen units: $M_\odot$).

In [ ]:
DATA_ROOT = "../../data/local_trees/environment_test"
CASES = ["early", "middle", "late"]
A_VALUES = (0.00, 0.12, 0.25, 0.37, 0.50, 0.62, 0.75)  # same sweep as build_environment_test.py
MODELS = ["fid"] + [f"A{a:.2f}" for a in A_VALUES]
MASS_THRESHOLD = 1e9  # Msun -- the z=0 mass floor for every plot below

# one consistent color per model, used in every panel of the SHMF and radial figures --
# a Blues ramp sampled across A_VALUES (darker = stronger scaling), fid kept as black/dashed
# so it stays visually distinct from the A-scaled trees at any sweep size
_blues = plt.cm.Blues(np.linspace(0.35, 0.95, len(A_VALUES)))
COLORS = {"fid": "black", **{f"A{a:.2f}": _blues[i] for i, a in enumerate(A_VALUES)}}
LINESTYLES = {"fid": "--", **{f"A{a:.2f}": "-" for a in A_VALUES}}

def tree_file(case, model):
    return f"{DATA_ROOT}/{case}/tree_{model}_evo.npz"

readers = {
    case: {
        model: jsh.Tree_Reader_Light(file=tree_file(case, model), mass_threshold=MASS_THRESHOLD)
        for model in MODELS
    }
    for case in CASES
}

for case in CASES:
    for model in MODELS:
        t = readers[case][model]
        print(f"{case:7s} {model:6s}  Rvir(z=0)={t.host_Rvir[0]:7.2f} kpc  "
              f"N(z0, m>{MASS_THRESHOLD:.0e})={int(t.z0_keep_mask.sum())}")

## Present-day subhalo mass function

$N$ per log-mass bin (`jsm_stats.count_straight`, same binning tool as `SHMF.ipynb`), 6 log-spaced bins from $10^{9}\,M_\odot$ up to the largest surviving subhalo across all 24 trees. One panel per case; all 8 models (fid + 7 A's) overlaid with the fixed color scheme above.

In [ ]:
# shared log-spaced mass bins across every panel, so panels are directly comparable
all_z0_masses = np.concatenate([
    readers[case][model].z0_mass[readers[case][model].z0_keep_mask]
    for case in CASES for model in MODELS
])
mass_bins = np.logspace(np.log10(MASS_THRESHOLD), np.log10(all_z0_masses.max() * 1.1), 7)  # 6 bins
bin_centers = np.sqrt(mass_bins[:-1] * mass_bins[1:])  # geometric mean, for log-log plotting

fig, axes = plt.subplots(1, 3, figsize=(double_textwidth, single_textwidth), sharex=True, sharey=True)

for ax, case in zip(axes, CASES):
    for model in MODELS:
        t = readers[case][model]
        masses = t.z0_mass[t.z0_keep_mask]
        counts = jsm_stats.count_straight(masses, mass_bins=mass_bins)
        ax.plot(bin_centers, counts, color=COLORS[model], ls=LINESTYLES[model],
                marker=".", label=model)
    ax.set_title(case)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("m$_{\\rm sub}$ [M$_\\odot$]")

axes[0].set_ylabel("N per mass bin")
axes[0].legend(framealpha=1, fontsize=7, ncol=2)

plt.tight_layout()
# plt.savefig("../../figures/environment_test_SHMF.pdf", bbox_inches="tight")  # uncomment to save
plt.show()

## Radial distribution of surviving subhalos

Cumulative $N(>r/R_{\rm vir})$ (`jsm_stats.radii_grthan`), same $m>10^{9}\,M_\odot$ z=0-surviving sample as above, radius scaled by each tree's own $R_{\rm vir}(z=0)$, counts on a log y-axis.

In [ ]:
r_bins = np.linspace(0, 1, 11)  # in units of R_vir(z=0)
r_bin_centers = (r_bins[:-1] + r_bins[1:]) / 2

fig, axes = plt.subplots(1, 3, figsize=(double_textwidth, single_textwidth), sharex=True, sharey=True)

for ax, case in zip(axes, CASES):
    for model in MODELS:
        t = readers[case][model]
        r_over_rvir = t.rmags_z0[t.z0_keep_mask] / t.host_Rvir[0]
        cum_counts = jsm_stats.radii_grthan(r_over_rvir, bins=r_bins)
        ax.plot(r_bin_centers, cum_counts, color=COLORS[model], ls=LINESTYLES[model],
                marker=".", label=model)
    ax.set_title(case)
    ax.set_xlabel("r / R$_{\\rm vir}$(z=0)")
    ax.set_xlim(0, 1)
    ax.set_yscale("log")

axes[0].set_ylabel("N(> r / R$_{\\rm vir}$)")
axes[0].legend(framealpha=1, fontsize=7, ncol=2)

plt.tight_layout()
# plt.savefig("../../figures/environment_test_radial.pdf", bbox_inches="tight")  # uncomment to save
plt.show()

## Nsub and fsub vs A

$N_{\rm sub}$ (count of z=0-surviving subhalos above `MASS_THRESHOLD`) and $f_{\rm sub} = \sum m_{\rm sub}(z{=}0) / M_{\rm host}(z{=}0)$ (same subhalo-mass-fraction definition as `Tree_Reader.compute_regimes`'s `fsub_matrix`), plotted against the scaling strength A for each case. One line per case (color, not model, since the x-axis here is A); `fid` isn't itself an A-scaled tree, so it's shown as a star at A=0 for reference rather than connected into the line.

In [ ]:
A_MODELS = [f"A{a:.2f}" for a in A_VALUES]  # excludes fid -- see note above
A_OF_MODEL = {f"A{a:.2f}": a for a in A_VALUES}

# distinct from the model color scheme above (this plot colors by case, not model)
CASE_COLORS = {"early": "#d62728", "middle": "#7f7f7f", "late": "#1f77b4"}

def nsub_fsub(t):
    Nsub = int(t.z0_keep_mask.sum())
    fsub = float(np.nansum(t.z0_mass) / t.target_mass)
    return Nsub, fsub

fig, axes = plt.subplots(1, 2, figsize=(double_textwidth, single_textwidth))

for case in CASES:
    A_vals = [A_OF_MODEL[m] for m in A_MODELS]
    Nsub_vals, fsub_vals = zip(*(nsub_fsub(readers[case][m]) for m in A_MODELS))

    axes[0].plot(A_vals, Nsub_vals, color=CASE_COLORS[case], marker="o", label=case)
    axes[1].plot(A_vals, fsub_vals, color=CASE_COLORS[case], marker="o", label=case)

    Nsub_fid, fsub_fid = nsub_fsub(readers[case]["fid"])
    axes[0].scatter([0], [Nsub_fid], color=CASE_COLORS[case], marker="*", s=140,
                     edgecolor="k", zorder=5)
    axes[1].scatter([0], [fsub_fid], color=CASE_COLORS[case], marker="*", s=140,
                     edgecolor="k", zorder=5)

axes[0].set_xlabel("A")
axes[0].set_ylabel("N$_{\\rm sub}$")
axes[0].set_title(f"N(m > {MASS_THRESHOLD:.0e} M$_\\odot$)")

axes[1].set_xlabel("A")
axes[1].set_ylabel("f$_{\\rm sub}$")
axes[1].set_title("subhalo mass fraction")

axes[0].legend(framealpha=1, fontsize=9, title="case  (\u2605 = fid, at A=0)")

plt.tight_layout()
# plt.savefig("../../figures/environment_test_Nsub_fsub_vs_A.pdf", bbox_inches="tight")  # uncomment to save
plt.show()

## Host mass accretion history vs the 1e13 sample mean

Each case's own host MAH (identical across its eight models -- vr scaling only touches subhalo orbits, not the host) against the population-mean MAH of the full 1000-tree 13.0 mass-bin ensemble (`SatGen/etc/mean_MAH/13.0_files_mean_MAH.npz`). The legend gives each case's `rat = (1+z50)/(1+<z50>)` -- the same ratio that sets the A-scaling strength for that tree (`<z50>` recomputed here from `host_z50` across the full N1000 ensemble, matching `z50_vr_scale_test` project notes: 0.9304).

X-axis is `log10(1+z)` with the axis reversed so time increases to the right (z=0/today at the right edge) -- standing redshift-axis convention, not specific to this figure.

In [ ]:
import h5py

N1000_H5 = "../../data/zhao/N1000/13.0_files.h5"
with h5py.File(N1000_H5, "r") as f:
    ensemble_z50 = np.array([f[k]["host_z50"][()] for k in f.keys()])
MEAN_Z50 = float(ensemble_z50.mean())

mean_mah = np.load("../../../SatGen/etc/mean_MAH/13.0_files_mean_MAH.npz")

fig, ax = plt.subplots(figsize=(single_textwidth, single_textwidth))

# redshift-axis convention: log10(1+z), time increasing to the right (so z
# decreases to the right, z=0/today at the right edge) -- see project memory
mean_log1pz = np.log10(1 + mean_mah["z"])
ax.plot(mean_log1pz, mean_mah["M"], color="black", ls="--", lw=1.5,
        label="13.0 sample mean")

for case in CASES:
    t = readers[case]["fid"]  # host MAH is the same across A-variants within a case
    rat = (1 + t.host_z50) / (1 + MEAN_Z50)
    ax.plot(np.log10(1 + t.redshift), t.host_MAH, color=CASE_COLORS[case], lw=1.5,
            label=f"{case} (rat={rat:.2f})")

ax.set_xlabel("log(1+z)")
ax.set_ylabel("M$_{\\rm host}$ [M$_\\odot$]")
ax.set_yscale("log")
ax.set_xlim(mean_log1pz.max(), 0)  # reversed: time increases to the right
ax.legend(framealpha=1, fontsize=9)

plt.tight_layout()
# plt.savefig("../../figures/environment_test_MAH.pdf", bbox_inches="tight")  # uncomment to save
plt.show()

## Notes

- `fid` and `A0.00` start from *identical* initial vr for first-order subhalos (m(A=0)=1 is a no-op scaling) -- but the two were evolved as separate `jsm_SubEvo`-style integrations, and that integrator draws a random ejection probability (`np.random.rand()`) for higher-order subhalo release at each timestep, so `fid` and `A0.00` can still diverge slightly by z=0. They are not expected to be bit-identical, just very close.
- Sample sizes are small (single realization per model, not a 1000-tree ensemble) -- treat these as a qualitative first look at the environment effect, not a statistically powered comparison.